# Imports set up rendering

In [ ]:
# Set environment variables FIRST (lightweight)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
os.environ["JAX_CAPTURED_CONSTANTS_REPORT_FRAMES"]="-1"

In [ ]:
# JAX setup (this is the slow import - takes ~60s on HPC)
from pathlib import Path
project_root = Path.cwd().parents[0]
import jax
jax.config.update("jax_compilation_cache_dir", (project_root / "tmp/jax_cache").as_posix())
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
try:
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
except AttributeError:
    pass
jax.config.update("jax_default_matmul_precision", "high")
num_gpus = jax.device_count(backend="gpu")
print(f"JAX initialized with {num_gpus} GPU(s)")

In [ ]:
# Import packages and set environment variables
%load_ext autoreload
%autoreload 2

import time
import pandas as pd
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from omegaconf import OmegaConf, DictConfig
from collections import namedtuple
from tqdm.auto import tqdm
from jax_tqdm import scan_tqdm

from sphinx_training.utils.path_utils import (
    register_custom_resolvers, load_config_and_override_paths, convert_dict_to_path
)
from sphinx_training.training import ckpt_utils
from sphinx_training.envs.fruitfly import state_generator_env
from sphinx_training.envs.fruitfly import imitation
from sphinx_training.utils.data_utils import ReferenceClips
from sphinx_training.utils.rollout_saver import Rollout, HDF5RolloutSaver


import functools
import mujoco
import hydra
import mediapy as media
from natsort import natsorted
from mujoco import mjx
from mujoco_playground._src import mjx_env
from mujoco_playground._src.mjx_env import get_qpos_ids
register_custom_resolvers()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# Plotting defaults
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.spines.right"] = "False"
plt.rcParams["axes.spines.top"] = "False"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.figsize"] = (3,1.5)
plt.rcParams['axes.linewidth'] = 0.25
plt.rcParams["xtick.major.size"] = 1.7
plt.rcParams["ytick.major.size"] = 1.7
plt.rcParams["xtick.major.width"] = 0.25
plt.rcParams["ytick.major.width"] = 0.25
plt.rcParams["figure.dpi"] = 200

SMALL_SIZE = 6
LABEL_SIZE = 7
AXLABEL_SIZE = 8
TITLE_SIZE = 10

plt.rcParams["font.family"] = "Arial" # "Arial"
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=AXLABEL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=LABEL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=TITLE_SIZE)  # fontsize of the figure title

# moduleColors = get_module_colors()

dnColor = "#51bbad"
excColor = "#d84a2e"
inhColor = "#026b85"
mnColor = "#267655"

lightDnColor = "#a7dcd5ff"
lightExcColor = "#ff9985"
lightInhColor = "#00acd6"
# connColormap = sns.blend_palette([inhColor,lightInhColor,"white",lightExcColor,excColor],as_cmap=True)


In [ ]:
# ── Comprehensive actuator color scheme (all 197 actuators) ──
import matplotlib.colors as mcolors
import re

# ── 1. Segment × Side colors (for per-leg plots) ──
LEG_COLORS = {
    'T1_left':  '#1b7a6e',  'T1_right': '#6fc4b8',
    'T2_left':  '#c47b1a',  'T2_right': '#edb96a',
    'T3_left':  '#7b3fa0',  'T3_right': '#b98ad4',
}

# ── 2. Functional muscle-group palette (proximal → distal) ──
# 12 joint-level categories shared across T1/T2/T3 + 4 non-leg regions
FUNC_COLORS = {
    # ── Coxa joint ──
    'coxa_promotor':      '#d84a2e',   # red          (T1:28*, T2:sternal_anterior*, T3:coxa_prom*)
    'coxa_remotor':       '#a63520',   # dark red      (T1:29, T2:posterior_adductor*, T3:coxa_rem*)
    'coxa_ant_rotator':   '#e8744c',   # salmon        (T1:31*, T2:32*, T3:coxa_prom_rot)
    'coxa_post_rotator':  '#b04e35',   # brick         (T1:32*, T2:32*, T3:coxa_promC)
    'coxa_adductor':      '#cc6644',   # terracotta    (T1:33, T2:large_posterior*, T3:med_coxa_add*)
    # ── Trochanter joint ──
    'troch_extensor':     '#026b85',   # teal-blue     (T1:34*, T2:34*, T3:tro_ext*)
    'troch_flexor':       '#0090b0',   # cyan          (T1:trochanter_flexor*, T2:c_t_r*, T3:tr_prom*)
    # ── Femur ──
    'femur_reductor':     '#267655',   # forest green  (T1/T2/T3:femur_flexor)
    # ── Tibia joint ──
    'tibia_extensor':     '#4ea84e',   # bright green  (*tibia_extension)
    'tibia_flexor':       '#8bc34a',   # lime          (*tibia_flexion, a_t_f*, tf*, tibia_acc*)
    # ── Tarsus ──
    'tarsal_tendon':      '#7b3fa0',   # purple        (*tarsal_flexor — long tendon)
    'tarsal_depress_lev': '#b06ec8',   # lavender      (*tarsal_extension, *tarsal_flexion)
    # ── Non-leg ──
    'head':               '#555555',   # grey
    'antenna':            '#e6a817',   # amber
    'wing':               '#3377cc',   # sky blue
    'abdomen':            '#886644',   # brown
}

# ── 3. Classification rules (order matters — first match wins) ──
_FUNC_RULES = [
    # Non-leg
    ('head',               lambda n: n.startswith('head')),
    ('antenna',            lambda n: 'antenna' in n),
    ('wing',               lambda n: n.startswith('wing')),
    ('abdomen',            lambda n: n.startswith('abdomen')),
    # Tarsus (before tibia — tarsal_flexion would otherwise match tibia_flexor)
    ('tarsal_tendon',      lambda n: 'tarsal_flexor' in n and 'extension' not in n and 'flexion' not in n),
    ('tarsal_depress_lev', lambda n: 'tarsal_extension' in n or 'tarsal_flexion' in n),
    # Tibia
    ('tibia_extensor',     lambda n: 'tibia_extension' in n),
    ('tibia_flexor',       lambda n: any(k in n for k in ['tibia_flexion', 'a_t_f', '_tf', 'tibia_acc'])),
    # Femur
    ('femur_reductor',     lambda n: 'femur_flexor' in n),
    # Trochanter
    ('troch_extensor',     lambda n: any(k in n for k in ['_34', 'tro_ext'])),
    ('troch_flexor',       lambda n: any(k in n for k in ['trochanter_flexor', 'c_t_r', 'tr_prom', 'post_tr_flex', 'mu_T2_29'])),
    # Coxa
    ('coxa_adductor',      lambda n: any(k in n for k in ['_33', 'med_coxa_add'])),
    ('coxa_post_rotator',  lambda n: any(k in n for k in ['_32', 'coxa_promC'])),
    ('coxa_ant_rotator',   lambda n: any(k in n for k in ['_31', 'coxa_prom_rot'])),
    ('coxa_remotor',       lambda n: any(k in n for k in ['mu_T1_29', 'posterior_adductor', 'coxa_rem'])),
    ('coxa_promotor',      lambda n: any(k in n for k in ['_28', 'sternal_anterior', 'coxa_prom'])),
]

def classify_actuator(name: str) -> str:
    """Return the functional group key for an actuator name."""
    for group, rule in _FUNC_RULES:
        if rule(name):
            return group
    return 'unknown'

def actuator_color(name: str, by: str = 'function') -> str:
    """Return a hex color for an actuator name.
    
    Args:
        name: actuator name, e.g. 'mu_T1_28a_left'
        by:   'function' → color by biological function (FUNC_COLORS)
              'leg'      → color by segment+side       (LEG_COLORS)
    """
    if by == 'leg':
        for key, clr in LEG_COLORS.items():
            seg, side = key.split('_')
            if seg in name and side in name:
                return clr
        return '#888888'
    else:  # 'function'
        group = classify_actuator(name)
        return FUNC_COLORS.get(group, '#888888')

# ── Preview swatch ──
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5), gridspec_kw={'width_ratios': [1, 2]})

# Left: leg colors
ax = axes[0]
for i, (label, clr) in enumerate(LEG_COLORS.items()):
    ax.barh(i, 1, color=clr, edgecolor='none', height=0.8)
    ax.text(0.5, i, label, ha='center', va='center', fontsize=7,
            color='white' if mcolors.rgb_to_hsv(mcolors.to_rgb(clr))[2] < 0.55 else 'black')
ax.set_xlim(0, 1); ax.set_ylim(-0.5, len(LEG_COLORS)-0.5)
ax.set_title('By leg (segment×side)', fontsize=8); ax.axis('off')

# Right: functional groups
ax = axes[1]
for i, (label, clr) in enumerate(FUNC_COLORS.items()):
    ax.barh(i, 1, color=clr, edgecolor='none', height=0.8)
    ax.text(0.5, i, label.replace('_', ' '), ha='center', va='center', fontsize=6,
            color='white' if mcolors.rgb_to_hsv(mcolors.to_rgb(clr))[2] < 0.55 else 'black')
ax.set_xlim(0, 1); ax.set_ylim(-0.5, len(FUNC_COLORS)-0.5)
ax.set_title('By biological function (16 groups)', fontsize=8); ax.axis('off')
ax.invert_yaxis()
axes[0].invert_yaxis()

plt.tight_layout()
print("✅ actuator_color(name, by='function'|'leg') — covers all 197 actuators")
print("   classify_actuator(name) → functional group key")

# Load cfg

In [ ]:
version = 'walk'
# base_dir = Path(f'/gscratch/portia/eabe/fly_neuromech/{version}')
base_dir = Path(f'/data2/users/eabe/fly_neuromech/{version}')
run_cfg_list = natsorted(list(Path(base_dir).rglob('run_config.yaml')))
for n, run_cfg in enumerate(run_cfg_list):
    temp = OmegaConf.load(run_cfg)
    print(f'{n}:', temp.version, run_cfg)

# ###### Load and update config with specified paths template ###### 
cfg_num = -1#15

# NEW APPROACH: Load config a"nd replace paths using workstation.yaml template
cfg = load_config_and_override_paths(
    config_path=run_cfg_list[cfg_num],
    new_paths_template="default",    # Use workstation.yaml for local paths
    config_dir=Path.cwd().parent / "configs",
    verbose=False
)

print(f'✅ Loaded experiment: {cfg_num}, {cfg.version}: {run_cfg_list[cfg_num]}')

# Convert string paths to Path objects and create directories
cfg.paths = convert_dict_to_path(cfg.paths)
print("✅ Successfully converted all paths to Path objects and created directories")

fig_dir = cfg.paths.fig_dir
# media.set_show_save_dir(fig_dir)

In [ ]:

reference_path = cfg.paths.data_dir/ f"datasets/{cfg.dataset.clip_idx}"
# reference_path = Path('/data2/users/eabe/fly_neuromech/data/datasets/Fruitfly_v2_muscles_stationary_1000hz_interp_padded2.h5')

# reference_path = cfg.paths.data_dir/ f"clips/{cfg.dataset.clip_idx}"
# reference_clips = ReferenceClips.from_path(reference_path, enable_jax=True)
print(f"✅ Loading reference data from {reference_path}")
reference_clips = ReferenceClips.from_path(reference_path, enable_jax=True)

## load model

In [ ]:
# cfg.muscles['global_defaults']['dyntype'] = 'none'
# muscles2 = OmegaConf.load('/home/eabe/Research/MyRepos/fly_neuromech/configs/muscles/data_derived.yaml')
# cfg.muscles = muscles2
cfg.dataset.env_args['iterations'] = 10
cfg.dataset.env_args['ls_iterations'] = 10
# Create environment
# env = ClosedLoopImitation(cfg, reference_clips)

# cfg.muscles.global_defaults.dyntype = 'none'

# Load clips into memory (required for JAX tracing)
env = imitation.Imitation(cfg, reference_clips=reference_clips)
env = state_generator_env.LatentStateWrapper(cfg, jax.random.PRNGKey(0), env, state_generator_env.Obs_Adapter())

# Check action space
print(f"Action size: {env.action_size}")  # Total trainable params
num_envs = reference_clips.clip_lengths.shape[0]
rng = jax.random.PRNGKey(0)
reset_rng, act_rng = jax.random.split(rng)
reset_rng = jax.random.split(reset_rng, num_envs)
reset_rng = jnp.zeros_like(reset_rng)
clip_idxs = jnp.arange(num_envs)
jit_reset = jax.jit(env.reset)
jit_forward = jax.jit(mjx.forward)
vmap_reset = jax.vmap(jit_reset)
vmap_forward = jax.vmap(jit_forward)

# Use Partial to bind W as a pytree leaf — prevents W (2.49GB) from being
# captured as a JIT constant, enabling compilation caching
# from jax.tree_util import Partial
# step_fn = Partial(env.step_with_W, env.W_device)
# jit_step = jax.jit(step_fn)
# vmap_step = jax.vmap(jit_step)

# Load Inference

In [ ]:

task_obs_size, prop_size = env._calc_obs_size()
ckpt_dir = cfg.paths.ckpt_dir
# ckpt_dir = Path('/data/users/eabe/biomech_model/Flybody/fly_imitation/flight/run_id=29793337/ckpt')
ckpt_files = natsorted([Path(f.path) for f in os.scandir(ckpt_dir) if f.is_dir()])
# ckpt_files = natsorted(list(ckpt_dir.rglob('*.pkl')))
if len(ckpt_files) > 0:
    restore_checkpoint = ckpt_files[0]  # Latest checkpoint
# ev_data = trainer.load_checkpoint(restore_checkpoint)
policy, policy_params = ckpt_utils.load_policy(restore_checkpoint)

print(f"✅ Loaded policy from {restore_checkpoint}")
jit_inference_fn = jax.jit(policy)
vmap_inference = jax.vmap(jit_inference_fn, in_axes=(0, None))
# action = jnp.full((num_envs, env.action_size), ev_data[1])
# action

In [ ]:
overwrite = True
# env.fixed_stim_amplitude=0.0

nframes = jnp.max(env.reference_clips.clip_lengths)
clip_list = jnp.arange(num_envs)
# Process and save the rollout data
policy_dir = cfg.paths.save_dir/'policy_data'
policy_dir.mkdir(parents=True, exist_ok=True)
policy_file = policy_dir/'rollout2.h5'

data = vmap_reset(rng=reset_rng,clip_idx=clip_idxs)

# Pre-extract constants for the scan body (avoid closing over env/self)
wing_joint_idxs = env._wing_joint_idxs
default_wing_pos = env._default_wing_pos
enable_flight = env.enable_flight


In [ ]:
def run_scan(data, rng):
    vmap_step_fn = jax.vmap(env.step)

    @scan_tqdm(int(nframes), tqdm_type="notebook")
    def scan_step(carry, _):
        data, rng = carry
        rng, action_rng = jax.random.split(rng)
        action, _ = vmap_inference(data.obs, action_rng)
        data = vmap_step_fn(data, action=action)

        step_out = {
            'qpos': data.data.qpos,
            'qvel': data.data.qvel,
            'ctrl': data.data.ctrl,
            'time': data.data.time,
            'obs': data.obs,
            'reward': data.reward,
            'done': data.done,
            'sensordata': data.data.sensordata,
            'info': {
                'action': data.info['action'],
                'site_xmat': data.data.site_xmat,
                'xmat': data.data.xmat,
                'prev_action': data.info['prev_action'],
                'reference_clip': data.info['reference_clip'],
                'start_frame': data.info['start_frame'],
                'truncated': data.info['truncated'],
            },
            'metrics': {key: data.metrics[key] for key in data.metrics},
        }
        return (data, rng), step_out

    return jax.lax.scan(scan_step, (data, rng), jnp.arange(int(nframes)))

jit_scan = jax.jit(run_scan)
rng = jax.random.PRNGKey(0)


In [ ]:

# Then save with the same one-liner:
(final_data, _), scan_out = jit_scan(data, rng)
saver = HDF5RolloutSaver.create_from_scan_output(scan_out, policy_file)
# saver = HDF5RolloutSaver(policy_file)


In [ ]:
camera = 'track1-fly' #env.mj_model.camera(0).name
add_ghost = True
add_labels = True
clip_idx = 13 #302 # 301
clip_length = env.reference_clips.clip_lengths[clip_idx]
single_clip = saver.load_rollout(clip_idx, clip_length=clip_length)

#### Adjust pose for better visualization (optional) — set wings and abdomen to default position
abd_idxs = [mjx_env.get_qpos_ids(env.mj_model, [joint.name for joint in env._spec.joints if ('abdomen' in joint.name)])]
new_qpos = single_clip.qpos.at[:, env._wing_joint_idxs].set(0.0)
new_qpos = new_qpos.at[:, abd_idxs].set(0.0)
single_clip = single_clip.replace(qpos=new_qpos)
scene_option = mujoco.MjvOption()
scene_option.sitegroup[:] = [1, 1, 1, 0, 1, 0]
scene_option.geomgroup[:] = [1, 1, 1, 0, 0, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_ACTUATOR] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_STATIC] = True
modify_scene_fns = []
rendered_frames = env.render_ghost(single_clip, clip_idx=clip_idx, camera=camera, scene_option=scene_option, height=512, width=512, video_path=None, add_labels=add_labels, add_ghost=add_ghost, modify_scene_fns=modify_scene_fns)

media.show_video(rendered_frames, fps=50) #50 for walking 20 for flying